# Lezione 33 — Tokenizer e generazione

> **Modulo:** Transformer e modello open · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 15 (tokenizzazione), Lezione 32 (blocco Transformer).
>
> Obiettivo pratico unico: costruire un **tokenizer** dal corpus delle memorie
> e un **ciclo di generazione autoregressiva** con un modellino bigram in
> NumPy, per capire meccanicamente greedy vs sampling con temperatura — prima
> di usare un modello vero (Gemma, Lezione 35).

## Parte 1 — Dal testo ai token e ritorno

Un modello generativo non vede lettere: vede **id di token**. Servono quindi
due mappe: `token -> id` (encode) e `id -> token` (decode). Aggiungiamo due
token speciali: `<bos>` (inizio) e `<eos>` (fine), cosi' il modello sa quando
iniziare e fermarsi.

La generazione e' **autoregressiva**: si parte da `<bos>`, si predice il token
successivo, lo si aggiunge alla sequenza, e si ripete finche' non esce `<eos>`
o si raggiunge un limite.

In [1]:
import numpy as np
import pandas as pd
import re
from pathlib import Path

train = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_train.csv')


def tokenizza(testo):
    return re.findall(r"[a-zA-Z]+", str(testo).lower())


# vocabolario dal corpus + token speciali
BOS, EOS = "<bos>", "<eos>"
vocab = [BOS, EOS]
visti = set(vocab)
for t in train['text']:
    for w in tokenizza(t):
        if w not in visti:
            visti.add(w)
            vocab.append(w)

stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
print("dimensione vocabolario:", len(vocab))
print("primi token:", vocab[:8])
print("encode di 'the user':", [stoi[w] for w in tokenizza('the user')])

dimensione vocabolario: 66
primi token: ['<bos>', '<eos>', 'the', 'user', 'likes', 'walking', 'meetings', 'prefers']
encode di 'the user': [2, 3]


## Parte 2 — Un modellino bigram

Il modello piu' semplice possibile: contiamo, per ogni token, quali token lo
seguono nel corpus. La probabilita' del token successivo dipende **solo** dal
token corrente (bigram). Non e' un Transformer — ma il *ciclo di generazione*
e' identico a quello di un modello grande: la differenza e' solo *come* si
calcolano le probabilita' del token successivo.

In [2]:
V = len(vocab)
counts = np.zeros((V, V))

for t in train['text']:
    ids = [stoi[BOS]] + [stoi[w] for w in tokenizza(t)] + [stoi[EOS]]
    for a, b in zip(ids[:-1], ids[1:]):
        counts[a, b] += 1.0

# +1 smoothing per non avere probabilita' zero, poi normalizzo per riga
probs = (counts + 1.0)
probs = probs / probs.sum(axis=1, keepdims=True)
print("matrice di transizione:", probs.shape, "(token corrente -> prossimo)")
print("ogni riga somma a 1:", np.allclose(probs.sum(axis=1), 1.0))

matrice di transizione: (66, 66) (token corrente -> prossimo)
ogni riga somma a 1: True


## Parte 3 — Greedy vs temperatura

Dalla distribuzione del prossimo token possiamo:

- **greedy**: prendere sempre il piu' probabile (deterministico, ripetitivo);
- **sampling con temperatura T**: campionare. $T<1$ rende la distribuzione piu'
  appuntita (piu' prudente), $T>1$ piu' piatta (piu' creativa/rischiosa).

La temperatura si applica ai logit: $p_i \propto \exp(\log p_i / T)$.

In [3]:
def applica_temperatura(p, T):
    logit = np.log(p + 1e-12) / T
    logit -= logit.max()
    e = np.exp(logit)
    return e / e.sum()


def genera(strategia="greedy", T=1.0, max_token=12, seed=0):
    rng = np.random.default_rng(seed)
    seq = [stoi[BOS]]
    while len(seq) < max_token:
        p = probs[seq[-1]]
        if strategia == "greedy":
            nxt = int(np.argmax(p))
        else:
            nxt = int(rng.choice(V, p=applica_temperatura(p, T)))
        if nxt == stoi[EOS]:
            break
        seq.append(nxt)
    return " ".join(itos[i] for i in seq[1:])  # salto <bos>


print("greedy        :", genera("greedy"))
print("temp T=0.7 #1 :", genera("sample", T=0.7, seed=1))
print("temp T=0.7 #2 :", genera("sample", T=0.7, seed=2))
print("temp T=1.3 #1 :", genera("sample", T=1.3, seed=1))

greedy        : the user prefers short updates about il progetto tensorflow
temp T=0.7 #1 : lucia lives tensorflow of bologna progetto giorgio had a for napoli
temp T=0.7 #2 : the user dislikes il colloquio long notifications
temp T=1.3 #1 : elena works for lives in dislikes every torino at


Leggi gli output: **greedy** produce sempre la stessa frase (il percorso piu'
probabile). Con la **temperatura** le frasi variano; T bassa resta vicina al
greedy, T alta rischia di piu'. E' esattamente il pomello che regolerai su un
modello vero nella Lezione 35 — solo che li' la distribuzione del prossimo
token viene da un Transformer, non da conteggi bigram.

## Parte 4 — Il progetto: Memory AI Lab, passo 33

Aggiungo al progetto un tokenizer riusabile (encode/decode con round-trip
garantito) e la funzione di generazione. Il tokenizer e' il pezzo che
alimentera' il modello open della Lezione 35.

In [4]:
def encode(testo):
    return [stoi[BOS]] + [stoi[w] for w in tokenizza(testo) if w in stoi] + [stoi[EOS]]


def decode(ids):
    return " ".join(itos[i] for i in ids if i not in (stoi[BOS], stoi[EOS]))


# controllo di non-regressione: round-trip e generazione valida
esempio = "the user likes walking meetings"
ids = encode(esempio)
assert decode(ids) == esempio, decode(ids)
assert all(w in vocab for w in genera('greedy').split())
print("round-trip encode/decode OK:", decode(ids))
print("token speciali nel vocab:", BOS in stoi and EOS in stoi)

round-trip encode/decode OK: the user likes walking meetings
token speciali nel vocab: True


## Riepilogo (max 8 punti)

1. Un modello generativo vede **id di token**, non testo: servono encode e
   decode.
2. I token speciali `<bos>`/`<eos>` segnano inizio e fine.
3. La generazione e' **autoregressiva**: predici, aggiungi, ripeti.
4. Un bigram predice il prossimo token dal solo token corrente.
5. Il *ciclo* di generazione e' identico a quello di un modello grande.
6. **Greedy**: sempre il piu' probabile → deterministico, ripetitivo.
7. **Temperatura**: $p_i\propto\exp(\log p_i/T)$; T bassa = prudente, alta =
   creativa.
8. Su Gemma (Lezione 35) cambia solo *come* si stima il prossimo token.

## Quiz

1. Perche' serve un token `<eos>`?
2. Cosa succede con temperatura T → 0?
3. Perche' il modello bigram non puo' "ricordare" l'inizio di una frase lunga?

*(Risposte: 1. per far fermare la generazione in modo appreso, non solo a
lunghezza fissa; 2. tende al greedy, sceglie il massimo; 3. perche' guarda solo
il token immediatamente precedente, nessun contesto piu' lungo — limite che
l'attenzione del Transformer supera.)*